# Offline WiFi research simulation

This notebook demonstrates the safe replay-oriented `gp_foundations.wifi_research` layer. It uses synthetic observations, neutral strategy labels, and a publication-friendly palette.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from gp_foundations.multioutput import CoregionalizationMatrix
from gp_foundations.wifi_research import JointStrategySimulator, ReplayEnvironment

strategy_ids = ('strategy_a', 'strategy_b', 'strategy_c')
grid = np.linspace(0.0, 1.0, 21)
colors = ['#0072B2', '#D55E00', '#009E73']

environment = ReplayEnvironment.synthetic(
    strategy_ids,
    grid,
    reward_functions={
        'strategy_a': lambda intensity: 0.95 - 3.0 * (intensity - 0.30) ** 2,
        'strategy_b': lambda intensity: 0.80 - 2.3 * (intensity - 0.70) ** 2,
        'strategy_c': lambda intensity: 0.72 - 1.6 * (intensity - 0.55) ** 2,
    },
    cost_functions={
        'strategy_a': lambda intensity: 0.6,
        'strategy_b': lambda intensity: 0.2,
        'strategy_c': lambda intensity: 0.1,
    },
    rounds_per_strategy=8,
    noise_std=0.03,
    rng=np.random.default_rng(7),
)

simulator = JointStrategySimulator(
    strategy_ids,
    grid,
    coregionalization=CoregionalizationMatrix.from_factor(
        np.array([[1.0, 0.0, 0.0], [0.65, 0.35, 0.0], [0.25, 0.15, 0.50]])
    ),
    slow_interval=4,
)
simulator.ingest_environment(environment)
evaluation = simulator.joint_grid_evaluation(rng=np.random.default_rng(11))
recommendations = simulator.evaluate_recommendations(
    rng=np.random.default_rng(11),
    cost_penalty=0.25,
    strategy_costs={'strategy_a': 0.6, 'strategy_b': 0.2, 'strategy_c': 0.1},
)

fig, ax = plt.subplots(figsize=(6.2, 3.6))
for idx, strategy_id in enumerate(strategy_ids):
    mean = evaluation.posterior.mean[:, idx]
    std = np.sqrt(evaluation.posterior.variance[:, idx])
    ax.plot(grid, mean, color=colors[idx], label=f'{strategy_id} mean')
    ax.fill_between(grid, mean - std, mean + std, color=colors[idx], alpha=0.16)

best = recommendations[0]
ax.scatter([best.intensity], [best.posterior_mean], color='black', s=45, zorder=5, label='top recommendation')
ax.set_xlabel('Intensity')
ax.set_ylabel('Reward')
ax.set_title('Offline joint recommendation from synthetic replay data')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(frameon=False, ncol=2)
plt.tight_layout()

for recommendation in recommendations:
    print(recommendation)

plt.show()
